# Phase 1: Data Cleaning and Merge

This notebook loads all 9 Olist Brazilian E-Commerce tables, inspects them, builds the target variable (`is_late`), preprocesses geolocation using zip-code centroids, aggregates multi-row tables to order level, merges everything into a single dataset, handles missing values, filters incomplete months, and exports the cleaned result.

In [1]:
import pandas as pd
import numpy as np
import os

RAW = '../datasets/raw/'
PROCESSED = '../datasets/processed/'
os.makedirs(PROCESSED, exist_ok=True)

## 1. Load and inspect all tables

In [2]:
orders = pd.read_csv(RAW + 'olist_orders_dataset.csv')
items = pd.read_csv(RAW + 'olist_order_items_dataset.csv')
payments = pd.read_csv(RAW + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(RAW + 'olist_order_reviews_dataset.csv')
products = pd.read_csv(RAW + 'olist_products_dataset.csv')
sellers = pd.read_csv(RAW + 'olist_sellers_dataset.csv')
customers = pd.read_csv(RAW + 'olist_customers_dataset.csv')
geolocation = pd.read_csv(RAW + 'olist_geolocation_dataset.csv')
cat_translation = pd.read_csv(RAW + 'product_category_name_translation.csv')

tables = {
    'orders': orders, 'items': items, 'payments': payments,
    'reviews': reviews, 'products': products, 'sellers': sellers,
    'customers': customers, 'geolocation': geolocation,
    'cat_translation': cat_translation
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")
    display(df.head(2))
    print()

orders: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00



items: (112650, 7)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.9,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.9,19.93



payments: (103886, 5)


...

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP



geolocation: (1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP



cat_translation: (71, 2)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories


In [3]:
# Missing values summary
missing = []
for name, df in tables.items():
    for col in df.columns:
        pct = df[col].isna().mean() * 100
        if pct > 0:
            missing.append({'table': name, 'column': col, 'missing_pct': round(pct, 2)})

missing_df = pd.DataFrame(missing).sort_values('missing_pct', ascending=False)
print(f"Columns with missing values: {len(missing_df)}")
display(missing_df)

Columns with missing values: 13


,table,column,missing_pct
3,reviews,review_comment_title,88.34
4,reviews,review_comment_message,58.70
2,orders,order_delivered_customer_date,2.98
5,products,product_category_name,1.85
6,products,product_name_lenght,1.85
7,products,product_description_lenght,1.85
8,products,product_photos_qty,1.85
1,orders,order_delivered_carrier_date,1.79
0,orders,order_approved_at,0.16
9,products,product_weight_g,0.01


## 2. Translate product categories to English

In [4]:
products = products.merge(cat_translation, on='product_category_name', how='left')
products.drop(columns='product_category_name', inplace=True)
products.rename(columns={'product_category_name_english': 'product_category'}, inplace=True)

print(f"Unique categories: {products['product_category'].nunique()}")
print(f"Missing category: {products['product_category'].isna().sum()}")
products['product_category'].value_counts().head(10)

Unique categories: 71
Missing category: 623


product_category
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64

## 3. Filter delivered orders and build target variable

Keep only `order_status == 'delivered'`. Add an 11:59:59 buffer to estimated delivery date to avoid false-late labels on same-day deliveries (from Repo 1). Target: `is_late = 1` if delivered after estimated date.

In [5]:
print(f"Order statuses:\n{orders['order_status'].value_counts()}\n")

# Parse timestamps
ts_cols = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in ts_cols:
    orders[col] = pd.to_datetime(orders[col])

# Keep delivered only
orders = orders[orders['order_status'] == 'delivered'].copy()
print(f"Delivered orders: {len(orders)}")

# Drop rows with null delivery date
orders = orders.dropna(subset=['order_delivered_customer_date'])
print(f"After dropping null delivery: {len(orders)}")

# Drop rows with impossible timestamps (approval before purchase)
orders = orders[orders['order_approved_at'] >= orders['order_purchase_timestamp']]
print(f"After dropping invalid timestamps: {len(orders)}")

# Add buffer to estimated delivery date to avoid false-late on same-day deliveries
orders['order_estimated_delivery_date'] += pd.Timedelta(hours=11, minutes=59, seconds=59)

# Build target
orders['is_late'] = (
    orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']
).astype(int)

late_rate = orders['is_late'].mean()
print(f"\nLate deliveries: {orders['is_late'].sum()} / {len(orders)} ({late_rate:.1%})")

Order statuses:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Delivered orders: 96478
After dropping null delivery: 96470
After dropping invalid timestamps: 96456

Late deliveries: 7645 / 96456 (7.9%)


## 4. Geolocation preprocessing

Multiple lat/lng entries per zip code. Use centroid (mean) per zip instead of `drop_duplicates` to average out GPS noise. Keep city/state from the first entry per zip.

In [6]:
print(f"Raw geolocation rows: {len(geolocation)}")
print(f"Unique zip codes: {geolocation['geolocation_zip_code_prefix'].nunique()}")

# Centroid lat/lng per zip
geo_coords = (
    geolocation
    .groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']]
    .mean()
    .reset_index()
)

# City/state from first entry per zip
geo_info = (
    geolocation
    .drop_duplicates(subset=['geolocation_zip_code_prefix'])
    [['geolocation_zip_code_prefix', 'geolocation_city', 'geolocation_state']]
)

geo_clean = geo_coords.merge(geo_info, on='geolocation_zip_code_prefix', how='left')
print(f"Cleaned geolocation: {geo_clean.shape}")
geo_clean.head()

Raw geolocation rows: 1000163
Unique zip codes: 19015
Cleaned geolocation: (19015, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.550190,-46.634024,sao paulo,SP
1,1002,-23.548146,-46.634979,sao paulo,SP
2,1003,-23.548994,-46.635731,sao paulo,SP
3,1004,-23.549799,-46.634757,sao paulo,SP
4,1005,-23.549456,-46.636733,sao paulo,SP


## 5. Aggregate order_items to order level

Each order can have multiple items. Aggregate to one row per `order_id`: total price, total freight, item count, heaviest product weight, and primary seller.

In [7]:
# Merge product info into items first (for weight/dimensions)
items_prod = items.merge(products, on='product_id', how='left')

items_agg = items_prod.groupby('order_id').agg(
    n_items=('order_item_id', 'count'),
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    max_weight_g=('product_weight_g', 'max'),
    max_length_cm=('product_length_cm', 'max'),
    max_height_cm=('product_height_cm', 'max'),
    max_width_cm=('product_width_cm', 'max'),
    product_photos_qty=('product_photos_qty', 'max'),
    primary_seller=('seller_id', 'first'),
    primary_product=('product_id', 'first'),
    primary_category=('product_category', 'first'),
).reset_index()

print(f"Items aggregated: {items_agg.shape}")
print(f"Multi-item orders: {(items_agg['n_items'] > 1).sum()} / {len(items_agg)}")
items_agg.head()

Items aggregated: (98666, 12)
Multi-item orders: 9803 / 98666


,order_id,n_items,total_price,total_freight,max_weight_g,max_length_cm,max_height_cm,max_width_cm,product_photos_qty,primary_seller,primary_product,primary_category
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,650.0,28.0,9.0,14.0,4.0,48436dade18ac8b2bce089ec2a041202,4244733e06e7ecb4970a6e2683c13e61,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,30000.0,50.0,30.0,40.0,2.0,dd7ddc04e1b6c2c614352b383efe2d36,e5f2d52b802189ee658865ca93d83a8f,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,3050.0,33.0,13.0,33.0,2.0,5b51032eddd242adc84c38acab88f23d,c777355d18b72b67abbeef9df44fd0fd,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,200.0,16.0,10.0,15.0,1.0,9d7a1d34a5052409006425275ba1c2b4,7634da152a4610f1595efa32f14722fc,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,3750.0,35.0,40.0,30.0,1.0,df560393f3a51e74553ab94004ba5c87,ac6c3623068f30de03045865e4e10089,garden_tools


## 6. Aggregate payments to order level

Multiple payment rows per order (e.g., credit card + voucher). Sum payment value, take max installments, and keep the most common payment type. Drop `not_defined` payment types.

In [8]:
payments = payments[~payments['payment_type'].isin(['not_defined'])]

pay_agg = payments.groupby('order_id').agg(
    payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
    payment_type=('payment_type', lambda x: x.mode().iloc[0])
).reset_index()

print(f"Payments aggregated: {pay_agg.shape}")
print(f"\nPayment types:\n{pay_agg['payment_type'].value_counts()}")
pay_agg.head()

Payments aggregated: (99437, 4)

Payment types:
payment_type
credit_card    76132
boleto         19784
voucher         1994
debit_card      1527
Name: count, dtype: int64


,order_id,payment_value,payment_installments,payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,credit_card


## 7. Merge pipeline

Merge order by order, checking shape after each join. All joins are LEFT on orders as the base table.

In [9]:
df = orders.copy()
print(f"Base (orders): {df.shape}")

# 1. Items (aggregated)
df = df.merge(items_agg, on='order_id', how='left')
print(f"+ items_agg: {df.shape}")

# 2. Customers
df = df.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
               'customer_city', 'customer_state']],
    on='customer_id', how='left'
)
print(f"+ customers: {df.shape}")

# 3. Seller info (from primary seller)
df = df.merge(
    sellers[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']],
    left_on='primary_seller', right_on='seller_id', how='left'
)
print(f"+ sellers: {df.shape}")

# 4. Payments (aggregated)
df = df.merge(pay_agg, on='order_id', how='left')
print(f"+ payments: {df.shape}")

# 5. Customer geolocation
df = df.merge(
    geo_clean.rename(columns={
        'geolocation_zip_code_prefix': 'customer_zip',
        'geolocation_lat': 'customer_lat',
        'geolocation_lng': 'customer_lng'
    })[['customer_zip', 'customer_lat', 'customer_lng']],
    left_on='customer_zip_code_prefix', right_on='customer_zip', how='left'
)
df.drop(columns='customer_zip', inplace=True)
print(f"+ customer geolocation: {df.shape}")

# 6. Seller geolocation
df = df.merge(
    geo_clean.rename(columns={
        'geolocation_zip_code_prefix': 'seller_zip',
        'geolocation_lat': 'seller_lat',
        'geolocation_lng': 'seller_lng'
    })[['seller_zip', 'seller_lat', 'seller_lng']],
    left_on='seller_zip_code_prefix', right_on='seller_zip', how='left'
)
df.drop(columns='seller_zip', inplace=True)
print(f"+ seller geolocation: {df.shape}")

print(f"\nFinal merged shape: {df.shape}")

Base (orders): (96456, 9)
+ items_agg: (96456, 20)
+ customers: (96456, 24)
+ sellers: (96456, 28)
+ payments: (96456, 31)
+ customer geolocation: (96456, 33)
+ seller geolocation: (96456, 35)

Final merged shape: (96456, 35)


## 8. Missing value handling

In [10]:
# Check missing values in merged dataset
null_counts = df.isnull().sum()
null_pct = (df.isnull().mean() * 100).round(2)
null_summary = pd.DataFrame({'count': null_counts, 'pct': null_pct})
print("Columns with missing values:")
display(null_summary[null_summary['count'] > 0].sort_values('pct', ascending=False))

Columns with missing values:


,count,pct
primary_category,1350,1.40
product_photos_qty,1331,1.38
customer_lat,264,0.27
customer_lng,264,0.27
seller_lat,215,0.22
seller_lng,215,0.22
max_weight_g,16,0.02
max_length_cm,16,0.02
max_height_cm,16,0.02
max_width_cm,16,0.02


In [11]:
# Category: fill missing with 'unknown'
df['primary_category'].fillna('unknown', inplace=True)

# Product weight/dimensions: fill with category median
for col in ['max_weight_g', 'max_length_cm', 'max_height_cm', 'max_width_cm', 'product_photos_qty']:
    cat_median = df.groupby('primary_category')[col].transform('median')
    df[col] = df[col].fillna(cat_median)
    # If still null (unknown category), fill with global median
    df[col].fillna(df[col].median(), inplace=True)

# Geolocation: fill missing lat/lng with state centroid
for prefix, lat_col, lng_col, state_col in [
    ('customer', 'customer_lat', 'customer_lng', 'customer_state'),
    ('seller', 'seller_lat', 'seller_lng', 'seller_state'),
]:
    state_lat = df.groupby(state_col)[lat_col].transform('median')
    state_lng = df.groupby(state_col)[lng_col].transform('median')
    df[lat_col] = df[lat_col].fillna(state_lat)
    df[lng_col] = df[lng_col].fillna(state_lng)

# Remaining nulls
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
print(f"Remaining null columns: {len(remaining)}")
if len(remaining) > 0:
    display(remaining)

C:\Users\Devyansh\AppData\Local\Temp\ipykernel_23384\2596794311.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['primary_category'].fillna('unknown', inplace=True)
C:\Users\Devyansh\AppData\Local\Temp\ipykernel_23384\2596794311.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

Remaining null columns: 4


order_delivered_carrier_date    1
payment_value                   1
payment_installments            1
payment_type                    1
dtype: int64

## 9. Filter incomplete months

Exclude early-launch months (Sep-Dec 2016, very few orders) and truncated end months (Sep-Oct 2018, incomplete data). This aligns with our temporal train/val/test split.

In [12]:
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')

# Monthly order counts
monthly = df.groupby('order_month').size()
print("Monthly order counts:")
print(monthly.to_string())

# Exclude incomplete months
exclude_months = ['2016-09', '2016-10', '2016-11', '2016-12', '2018-09', '2018-10']
exclude_periods = [pd.Period(m, freq='M') for m in exclude_months]

before = len(df)
df = df[~df['order_month'].isin(exclude_periods)]
print(f"\nDropped {before - len(df)} orders from incomplete months")
print(f"Remaining: {len(df)} orders ({df['order_month'].min()} to {df['order_month'].max()})")

df.drop(columns='order_month', inplace=True)

Monthly order counts:
order_month
2016-09       1
2016-10     265
2016-12       1
2017-01     748
2017-02    1641
2017-03    2546
2017-04    2303
2017-05    3545
2017-06    3135
2017-07    3872
2017-08    4193
2017-09    4150
2017-10    4478
2017-11    7288
2017-12    5513
2018-01    7069
2018-02    6555
2018-03    7003
2018-04    6798
2018-05    6749
2018-06    6096
2018-07    6156
2018-08    6351
Freq: M

Dropped 267 orders from incomplete months
Remaining: 96189 orders (2017-01 to 2018-08)


## 10. Compute Haversine distance (seller to customer)

In [13]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    """Vectorized haversine distance in km."""
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine_vectorized(
    df['seller_lat'], df['seller_lng'],
    df['customer_lat'], df['customer_lng']
)

print(f"Distance stats (km):")
print(df['distance_km'].describe().round(1))
print(f"\nMissing distance: {df['distance_km'].isna().sum()}")

Distance stats (km):
count    96189.0
mean       601.5
std        593.3
min          0.0
25%        187.7
50%        435.2
75%        800.8
max       8677.9
Name: distance_km, dtype: float64

Missing distance: 0


## 11. Compute delivery time columns

In [14]:
# Actual delivery days (purchase to delivery)
df['actual_delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.total_seconds() / 86400

# Estimated delivery days (purchase to estimated)
df['estimated_delivery_days'] = (
    df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
).dt.total_seconds() / 86400

# Delivery delay (positive = late, negative = early)
df['delivery_delay_days'] = df['actual_delivery_days'] - df['estimated_delivery_days']

# Same-state flag
df['same_state'] = (df['customer_state'] == df['seller_state']).astype(int)

print("Delivery time stats (days):")
print(df[['actual_delivery_days', 'estimated_delivery_days', 'delivery_delay_days']].describe().round(1))
print(f"\nSame-state orders: {df['same_state'].mean():.1%}")

Delivery time stats (days):
       actual_delivery_days  estimated_delivery_days  delivery_delay_days
count               96189.0                  96189.0              96189.0
mean                   12.5                     24.1                -11.6
std                     9.5                      8.6                 10.1
min                     0.5                      2.5               -146.5
25%                     6.8                     18.8                -16.7
50%                    10.2                     23.7                -12.3
75%                    15.7                     28.9                 -6.9
max                   209.6                    155.6                188.5

Same-state orders: 36.0%


## 12. Drop unnecessary columns and export

In [15]:
# Drop ID columns not needed for modeling (keep order_id as index key)
drop_cols = ['customer_id', 'customer_unique_id', 'order_status',
             'primary_product', 'primary_seller', 'seller_id']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

# Drop post-delivery columns (leakage risk for modeling)
# Keep order_delivered_customer_date only for reference, not as a feature
# review_score is NOT included (post-delivery)

print(f"Final dataset: {df.shape}")
print(f"Late rate: {df['is_late'].mean():.2%}")
print(f"\nColumn list:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col} ({df[col].dtype})")

Final dataset: (96189, 34)
Late rate: 7.94%

Column list:
   1. order_id (object)
   2. order_purchase_timestamp (datetime64[ns])
   3. order_approved_at (datetime64[ns])
   4. order_delivered_carrier_date (datetime64[ns])
   5. order_delivered_customer_date (datetime64[ns])
   6. order_estimated_delivery_date (datetime64[ns])
   7. is_late (int32)
   8. n_items (int64)
   9. total_price (float64)
  10. total_freight (float64)
  11. max_weight_g (float64)
  12. max_length_cm (float64)
  13. max_height_cm (float64)
  14. max_width_cm (float64)
  15. product_photos_qty (float64)
  16. primary_category (object)
  17. customer_zip_code_prefix (int64)
  18. customer_city (object)
  19. customer_state (object)
  20. seller_zip_code_prefix (int64)
  21. seller_city (object)
  22. seller_state (object)
  23. payment_value (float64)
  24. payment_installments (float64)
  25. payment_type (object)
  26. customer_lat (float64)
  27. customer_lng (float64)
  28. seller_lat (float64)
  29. seller_l

In [16]:
df.to_csv(PROCESSED + 'olist_merged_clean.csv', index=False)
print(f"Saved to {PROCESSED}olist_merged_clean.csv")
print(f"Shape: {df.shape}")
print(f"Date range: {df['order_purchase_timestamp'].min()} to {df['order_purchase_timestamp'].max()}")

Saved to ../datasets/processed/olist_merged_clean.csv
Shape: (96189, 34)
Date range: 2017-01-05 11:56:06 to 2018-08-29 15:00:37


## Summary

- Loaded 9 Olist tables, translated categories to English
- Filtered to delivered orders, added 11:59:59 buffer to estimated delivery date
- Used zip-code centroid for geolocation (averaging GPS noise)
- Aggregated items and payments to order level
- Merged into single dataset via LEFT JOINs
- Filled missing values: category median for product dimensions, state centroid for geolocation
- Excluded incomplete months (Sep-Dec 2016, Sep-Oct 2018)
- Computed haversine distance, delivery time columns, same-state flag
- Output: `datasets/processed/olist_merged_clean.csv`